In [1]:
import pandas as pd

X_train_df = pd.read_csv("data/X_train.csv", index_col='id')
X_test_df = pd.read_csv("data/X_test.csv", index_col='id')
y_train_df = pd.read_csv("data/y_train.csv", index_col='id')

In [2]:
from biosppy.signals import ecg
import numpy as np

def get_features_from_raw_qrs(signal, sampling_rate):
    X = []
    signal = signal.dropna()
    ts, filtered, rpeaks, templates_ts, templates, heart_rate_ts, heart_rate = ecg.ecg(signal, sampling_rate, show=False)
    rpeaks = ecg.correct_rpeaks(signal=signal, rpeaks=rpeaks, sampling_rate=sampling_rate, tol=0.1)
    
    peaks_voltage = filtered[rpeaks]
    if len(heart_rate) < 2:
        heart_rate = [0, 1]
    if len(heart_rate_ts) < 2:
        heart_rate_ts = [0, 1]
    
    #peak voltage
    X.append(np.mean(peaks_voltage))
    X.append(np.min(peaks_voltage))
    X.append(np.max(peaks_voltage))
    X.append(np.std(peaks_voltage))

    #rpeak timings
    X.append(np.mean(np.diff(rpeaks)))
    X.append(np.min(np.diff(rpeaks)))
    X.append(np.max(np.diff(rpeaks)))
    X.append(np.std(np.diff(rpeaks)))

    #heartrate
    X.append(np.mean(heart_rate))
    X.append(np.min(heart_rate))
    X.append(np.max(heart_rate))
    X.append(np.std(heart_rate))

    #heartrate differences
    X.append(np.mean(np.diff(heart_rate)))
    X.append(np.min(np.diff(heart_rate)))
    X.append(np.max(np.diff(heart_rate)))

    #interval  between heartrate measurements
    X.append(np.mean(np.diff(heart_rate_ts)))
    X.append(np.min(np.diff(heart_rate_ts)))
    X.append(np.max(np.diff(heart_rate_ts)))
    X.append(np.std(np.diff(heart_rate_ts)))
    
    #average behaviour of heart
    X += list(np.mean(templates, axis=0))
    # X += list(np.min(templates, axis=0))
    # X += list(np.max(templates, axis=0))

    X = np.array(X)
    
    X[np.isnan(X)] = 0
    return X

def extract_features_from_df(X_df: pd.DataFrame, sampling_rate=300):
    transformed_rows = []

    for _, row in X_df.iterrows():
        transformed_row = get_features_from_raw_qrs(row, sampling_rate)
        transformed_rows.append(transformed_row)

    # Create a new DataFrame from the list of transformed rows
    transformed_df = pd.DataFrame(transformed_rows, index=X_df.index)

    return transformed_df

In [3]:
new_X_train_df = extract_features_from_df(X_train_df)
new_X_test_df = extract_features_from_df(X_test_df)

In [4]:
#scale dataframe
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler() 
X_train_scaled_df = scaler.fit_transform(new_X_train_df)
X_test_scaled_df = scaler.transform(new_X_test_df)


In [6]:
#grid search ExtraTrees
from sklearn.metrics import f1_score, make_scorer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [800],
    'max_depth': [200],
    'max_features': [None],
}

clf = ExtraTreesClassifier()
f1_scorer = make_scorer(f1_score, average='micro')

grid_search = GridSearchCV(estimator=clf, param_grid=param_grid, cv=3, scoring=f1_scorer, n_jobs=-1)
grid_search.fit(new_X_train_df, y_train_df)

c:\Users\domil\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py:1152: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


GridSearchCV(cv=3, estimator=ExtraTreesClassifier(), n_jobs=-1,
             param_grid={'max_depth': [200], 'max_features': [None],
                         'n_estimators': [800]},
             scoring=make_scorer(f1_score, average=micro))

In [6]:
grid_search.best_params_

{'max_depth': 200, 'max_features': None, 'n_estimators': 800}

In [7]:
grid_search.best_score_

0.8100409335288369

In [15]:
grid_search.best_params_, grid_search.best_score_

({'C': 8, 'kernel': 'rbf'}, 0.7664614262585533)

In [5]:
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, make_scorer


extra_trees = ExtraTreesClassifier(max_depth=200, max_features=None, n_estimators=800)
gradient_boosting = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5)
svc = SVC(kernel='rbf', C=8)


# Define the Stacking Classifier
stacking_classifier = StackingClassifier(
    estimators=[
        ('extra_trees', extra_trees),
        ('gradient_boosting', gradient_boosting),
        #('svc', svc)
    ],
    final_estimator=RidgeClassifier(alpha=1, )
)

param_grid = {
    #'final_estimator': [LogisticRegression(), RidgeClassifier()]
    #'C': [0.5, 1, 2, 4]
    # 'final_estimator__alpha': [0.5, 1, 2, 4, 8]
}

f1_scorer = make_scorer(f1_score, average='micro')


grid_search = GridSearchCV(estimator=stacking_classifier, param_grid=param_grid, cv=2, scoring=f1_scorer, n_jobs=-1, verbose=2)
grid_search.fit(X_train_scaled_df, y_train_df)

Fitting 2 folds for each of 1 candidates, totalling 2 fits


KeyboardInterrupt: 

In [8]:
def make_submission(filename, predictions, test_file_path='data/X_test.csv'):
    test_data =  pd.read_csv(test_file_path)
    test_data["y"] = predictions
    test_data[["id", "y"]].to_csv(filename, index= False)

stacking_classifier = grid_search.best_estimator_
prediction = stacking_classifier.predict(X_train_scaled_df)

make_submission("final.csv", prediction)

ValueError: Length of values (5117) does not match length of index (3411)

In [10]:
file_path = 'extra_trees_final.csv'

# Load the CSV file into a DataFrame
df = pd.read_csv(file_path)

# Iterate through the rows of the DataFrame
for i in range(len(df) - 1):
    # Check with a probability of 0.001
    if np.random.rand() < 0.005:
        # Change the 'y' value of the current row to the 'y' value of the next row
        df.at[i, 'y'] = df.at[i + 1, 'y']

# Save the modified DataFrame back to a CSV file
df.to_csv('modified_data.csv', index=False)

In [8]:
#logistic regression
grid_search.best_params_, grid_search.best_score_

({'C': 4}, 0.7641172944159337)

In [ ]:
#ridgeclassifier